[nvidia sortformer](https://docs.nvidia.com/nemo-framework/user-guide/24.09/nemotoolkit/core/export.html) 

In [ ]:
import torch
import numpy as np
import onnx
import onnxruntime as ort

from nemo.collections.asr.models import SortformerEncLabelModel

MODEL_NAME = "nvidia/diar_streaming_sortformer_4spk-v2.1"

model = SortformerEncLabelModel.from_pretrained(
    model_name=MODEL_NAME,
)

model.eval()
model.cpu()

print("loaded")
print(type(model))

OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


RepositoryNotFoundError: 404 Client Error. (Request ID: Root=1-6a9f4238-09713cbc2169a27d7b5b9efd;b5f256d2-80ce-413a-a986-b37d2974227d)

Repository Not Found for url: https://huggingface.co/api/models/nvidia/diar_sortformer_4spk-v2?blobs=true.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication

### INSPECT

In [3]:
print("INPUT TYPES")
print(model.input_types)

print()
print("OUTPUT TYPES")
print(model.output_types)


print(model.forward)
print(model.forward_infer)

print(model.preprocessor)
print("Sample rate:", model.preprocessor._sample_rate)

INPUT TYPES
{'audio_signal': NeuralType(axis=(batch, time), element_type=AudioSignal), 'audio_signal_length': NeuralType(axis=(batch,), element_type=LengthsType)}

OUTPUT TYPES
OrderedDict([('preds', NeuralType(axis=(batch, time, dimension), element_type=ProbsType))])
<bound method SortformerEncLabelModel.forward of SortformerEncLabelModel(
  (preprocessor): AudioToMelSpectrogramPreprocessor(
    (featurizer): FilterbankFeatures()
  )
  (encoder): ConformerEncoder(
    (pre_encode): ConvSubsampling(
      (out): Linear(in_features=2560, out_features=512, bias=True)
      (conv): MaskedConvSequential(
        (0): Conv2d(1, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (1): ReLU(inplace=True)
        (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=256)
        (3): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
        (4): ReLU(inplace=True)
        (5): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=256

### Check model works

In [4]:
sample_rate = int(model.preprocessor._sample_rate)
duration = 10

num_samples = sample_rate * duration

audio = torch.randn(
    1,
    num_samples,
    dtype=torch.float32,
)

audio_length = torch.tensor(
    [num_samples],
    dtype=torch.long,
)

print("Sample Rate:", sample_rate, num_samples)
print(audio.shape)
print(audio_length)

with torch.inference_mode():
    torch_output = model(
        audio_signal=audio,
        audio_signal_length=audio_length,
    )

print("shape:", torch_output.shape)
print("dtype:", torch_output.dtype)

Sample Rate: 16000 160000
torch.Size([1, 160000])
tensor([160000])
shape: torch.Size([1, 125, 4])
dtype: torch.float32


### exports

In [5]:
OUTPUT = "../../models/sortformer.onnx"

model.export(
    OUTPUT,
    input_example=(
        audio,
        audio_length,
    ),
    onnx_opset_version=17,
    do_constant_folding=True,
    dynamic_axes={
        "audio_signal": {
            0: "batch",
            1: "samples",
        },
        "audio_signal_length": {
            0: "batch",
        },
        "preds": {
            0: "batch",
            1: "frames",
        },
    },
)

print(f"Exported to {OUTPUT}")

TypeError: SortformerEncLabelModel.forward_for_export() missing 4 required positional arguments: 'spkcache', 'spkcache_lengths', 'fifo', and 'fifo_lengths'

### Validate

In [ ]:
onnx_model = onnx.load(OUTPUT)

onnx.checker.check_model(onnx_model)

print("ONNX model is valid")

session = ort.InferenceSession(
    OUTPUT,
    providers=["CPUExecutionProvider"],
)

print("INPUTS")

for x in session.get_inputs():
    print(
        f"{x.name}: "
        f"shape={x.shape}, "
        f"type={x.type}"
    )

print()
print("OUTPUTS")

for x in session.get_outputs():
    print(
        f"{x.name}: "
        f"shape={x.shape}, "
        f"type={x.type}"
    )

onnx_output = session.run(
    None,
    {
        "audio_signal": audio.numpy(),
        "audio_signal_length": audio_length.numpy(),
    },
)[0]

print("ONNX output shape:", onnx_output.shape)

pytorch_output = torch_output.detach().cpu().numpy()

max_error = np.max(
    np.abs(pytorch_output - onnx_output)
)

mean_error = np.mean(
    np.abs(pytorch_output - onnx_output)
)

print("PyTorch:", pytorch_output.shape)
print("ONNX:   ", onnx_output.shape)
print()
print("max error: ", max_error)
print("mean error:", mean_error)

for node in onnx_model.graph.node[:30]:
    print(node.op_type, node.name)

np.testing.assert_allclose(
    pytorch_output,
    onnx_output,
    rtol=1e-3,
    atol=1e-4,
)

print("✅ PyTorch and ONNX match")

### Processor

In [1]:
import math
import inspect
from nemo.collections.asr.models import SortformerEncLabelModel

# The model should already be loaded:
#
model = SortformerEncLabelModel.from_pretrained(
    "nvidia/diar_streaming_sortformer_4spk-v2"
)

featurizer = model.preprocessor.featurizer

print("=" * 70)
print("SORTFORMER V2 — ACTUAL NE-MO FBANK CONFIG")
print("=" * 70)

# ------------------------------------------------------------
# 1. Actual FilterbankFeatures attributes
# ------------------------------------------------------------

attrs = [
    "sample_rate",
    "win_length",
    "hop_length",
    "n_fft",
    "window",
    "normalize",
    "preemph",
    "nfilt",
    "lowfreq",
    "highfreq",
    "log",
    "log_zero_guard_type",
    "log_zero_guard_value",
    "dither",
    "pad_to",
    "frame_splicing",
    "exact_pad",
    "pad_value",
    "mag_power",
    "nb_augmentation_prob",
    "nb_max_freq",
    "mel_norm",
]

print("\n[FilterbankFeatures]")
for name in attrs:
    if hasattr(featurizer, name):
        value = getattr(featurizer, name)

        # Tensor values can be huge, don't print them here.
        if name == "window":
            print(f"{name:25} = Tensor(shape={tuple(value.shape)})")
        elif name == "fb":
            print(f"{name:25} = <mel filterbank>")
        else:
            print(f"{name:25} = {value}")

# ------------------------------------------------------------
# 2. Preprocessor-level configuration
# ------------------------------------------------------------

print("\n[AudioToMelSpectrogramPreprocessor]")

preprocessor_attrs = [
    "_sample_rate",
    "win_length",
    "hop_length",
]

for name in preprocessor_attrs:
    if hasattr(model.preprocessor, name):
        print(f"{name:25} = {getattr(model.preprocessor, name)}")

# ------------------------------------------------------------
# 3. Model config
# ------------------------------------------------------------

print("\n[Model Config: preprocessor]")

try:
    pre_cfg = model.cfg.preprocessor

    # OmegaConf configs
    try:
        from omegaconf import OmegaConf
        print(OmegaConf.to_yaml(pre_cfg))
    except Exception:
        print(pre_cfg)

except Exception as e:
    print(f"Could not read model.cfg.preprocessor: {e}")

# ------------------------------------------------------------
# 4. Derived values useful for Go
# ------------------------------------------------------------

print("\n[Derived values]")

sample_rate = featurizer.sample_rate
win_length = featurizer.win_length
hop_length = featurizer.hop_length
n_fft = featurizer.n_fft
n_mels = featurizer.nfilt

print(f"{'sample_rate':25} = {sample_rate} Hz")
print(f"{'window_size':25} = {win_length / sample_rate:.6f} sec")
print(f"{'window_size_samples':25} = {win_length}")
print(f"{'window_stride':25} = {hop_length / sample_rate:.6f} sec")
print(f"{'window_stride_samples':25} = {hop_length}")
print(f"{'n_fft':25} = {n_fft}")
print(f"{'fft_bins':25} = {n_fft // 2 + 1}")
print(f"{'n_mels':25} = {n_mels}")

if hasattr(featurizer, "lowfreq"):
    print(f"{'lowfreq':25} = {featurizer.lowfreq}")

if hasattr(featurizer, "highfreq"):
    print(f"{'highfreq':25} = {featurizer.highfreq}")

# ------------------------------------------------------------
# 5. Actual learned/constructed Mel filterbank
# ------------------------------------------------------------

print("\n[Mel Filterbank Tensor]")

if hasattr(featurizer, "fb"):
    fb = featurizer.fb

    print(f"shape = {tuple(fb.shape)}")
    print(f"dtype = {fb.dtype}")
    print(f"device = {fb.device}")

    # The actual numbers can be saved for exact Go reproduction.
    import numpy as np

    np.save(
        "sortformer_v2_mel_filterbank.npy",
        fb.detach().cpu().numpy(),
    )

    print("saved = sortformer_v2_mel_filterbank.npy")

# ------------------------------------------------------------
# 6. Actual window function
# ------------------------------------------------------------

print("\n[Window Tensor]")

if hasattr(featurizer, "window"):
    window = featurizer.window

    print(f"shape = {tuple(window.shape)}")
    print(f"dtype = {window.dtype}")

    import numpy as np

    np.save(
        "sortformer_v2_window.npy",
        window.detach().cpu().numpy(),
    )

    print("saved = sortformer_v2_window.npy")

# ------------------------------------------------------------
# 7. Streaming model settings
# ------------------------------------------------------------

print("\n[Sortformer Streaming Config]")

streaming_attrs = [
    "chunk_len",
    "chunk_left_context",
    "chunk_right_context",
    "fifo_len",
    "spkcache_len",
    "spkcache_update_period",
]

modules = getattr(model, "sortformer_modules", None)

if modules is not None:
    for name in streaming_attrs:
        if hasattr(modules, name):
            print(f"{name:25} = {getattr(modules, name)}")

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo W 2026-09-07 11:25:44 sortformer_diar_models:115] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    num_spks: 4
    session_len_sec: 90
    soft_label_thres: 0.5
    soft_targets: false
    labels: null
    batch_size: 4
    shuffle: true
    num_workers: 18
    validation_mode: false
    use_lhotse: false
    use_bucketing: false
    pin_memory: true
    window_stride: 0.01
    subsampling_factor: 8
    
[NeMo W 2026-09-07 11:25:44 sortformer_diar_models:115] If you intend to do validation, please call the ModelPT.setup_

[NeMo I 2026-09-07 11:25:44 modelPT:489] Model SortformerEncLabelModel was successfully restored from /home/arthur/.cache/huggingface/hub/models--nvidia--diar_streaming_sortformer_4spk-v2/snapshots/5240a64075176943f677d30fa2171c780229f341/diar_streaming_sortformer_4spk-v2.nemo.
SORTFORMER V2 — ACTUAL NE-MO FBANK CONFIG

[FilterbankFeatures]
sample_rate               = 16000
win_length                = 400
hop_length                = 160
n_fft                     = 512
window                    = Tensor(shape=(400,))
normalize                 = NA
preemph                   = 0.97
nfilt                     = 128
log                       = True
log_zero_guard_type       = add
log_zero_guard_value      = 5.960464477539063e-08
dither                    = 1e-05
pad_to                    = 16
frame_splicing            = 1
exact_pad                 = False
pad_value                 = 0
mag_power                 = 2.0
nb_augmentation_prob      = 0.0

[AudioToMelSpectrogramPreprocessor]
_sample